In [2]:
import pandas as pd
import numpy as np

print(pd.__version__)

3.0.3


In [3]:
restaurants = pd.read_csv("../data/raw/fast_food_restaurants.csv")
cities = pd.read_csv("../data/raw/city_population.csv")
states = pd.read_csv("../data/raw/state_population.csv")

In [4]:
print(cities.columns.tolist())
print(states.columns.tolist())

['city', 'city_ascii', 'state_id', 'state_name', 'county_fips', 'county_name', 'lat', 'lng', 'population', 'density', 'source', 'military', 'incorporated', 'timezone', 'ranking', 'zips', 'id']
['Label (Grouping)', 'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'Delaware', 'District of Columbia', 'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming', 'Puerto Rico']


In [6]:
states.head()


,Label (Grouping),Alabama,Alaska,Arizona,Arkansas,California,Colorado,Connecticut,Delaware,District of Columbia,...,Tennessee,Texas,Utah,Vermont,Virginia,Washington,West Virginia,Wisconsin,Wyoming,Puerto Rico
0,Total,"5,024,279","733,391","7,151,502","3,011,524","39,538,223","5,773,714","3,605,944","989,948","689,545",...,"6,910,840","29,145,505","3,271,616","643,077","8,631,393","7,705,281","1,793,716","5,893,718","576,851","3,285,874"


In [7]:
states.shape


(1, 53)

In [8]:
restaurants.columns.tolist()

['address',
 'city',
 'country',
 'keys',
 'latitude',
 'longitude',
 'name',
 'postalCode',
 'province',
 'websites']

In [9]:
states_long = states.melt(
    id_vars=["Label (Grouping)"],
    var_name="state_name",
    value_name="state_population"
)

states_long = states_long[
    states_long["state_name"] != "Puerto Rico"
]

states_long.head()

,Label (Grouping),state_name,state_population
0,Total,Alabama,"5,024,279"
1,Total,Alaska,"733,391"
2,Total,Arizona,"7,151,502"
3,Total,Arkansas,"3,011,524"
4,Total,California,"39,538,223"


In [10]:
states_long["state_population"] = (
    states_long["state_population"]
    .str.replace(",", "", regex=False)
    .astype(int)
)

states_long.head()

,Label (Grouping),state_name,state_population
0,Total,Alabama,5024279
1,Total,Alaska,733391
2,Total,Arizona,7151502
3,Total,Arkansas,3011524
4,Total,California,39538223


In [11]:
states_long.shape

(51, 3)

In [12]:
restaurants_clean = restaurants[
    [
        "name",
        "city",
        "province",
        "latitude",
        "longitude"
    ]
].copy()

In [13]:
restaurants_clean.head()

,name,city,province,latitude,longitude
0,McDonald's,Massena,NY,44.92130,-74.89021
1,Wendy's,Washington Court House,OH,39.53255,-83.44526
2,Frisch's Big Boy,Maysville,KY,38.62736,-83.79141
3,McDonald's,Massena,NY,44.95008,-74.84553
4,OMG! Rotisserie,Athens,OH,39.35155,-82.09728


In [14]:
restaurants_clean["city"] = (
    restaurants_clean["city"]
    .str.strip()
    .str.title()
)

cities["city"] = (
    cities["city"]
    .str.strip()
    .str.title()
)

In [15]:
state_map = {
    "AL":"Alabama",
    "AK":"Alaska",
    "AZ":"Arizona",
    "AR":"Arkansas",
    "CA":"California",
    "CO":"Colorado",
    "CT":"Connecticut",
    "DE":"Delaware",
    "DC":"District of Columbia",
    "FL":"Florida",
    "GA":"Georgia",
    "HI":"Hawaii",
    "ID":"Idaho",
    "IL":"Illinois",
    "IN":"Indiana",
    "IA":"Iowa",
    "KS":"Kansas",
    "KY":"Kentucky",
    "LA":"Louisiana",
    "ME":"Maine",
    "MD":"Maryland",
    "MA":"Massachusetts",
    "MI":"Michigan",
    "MN":"Minnesota",
    "MS":"Mississippi",
    "MO":"Missouri",
    "MT":"Montana",
    "NE":"Nebraska",
    "NV":"Nevada",
    "NH":"New Hampshire",
    "NJ":"New Jersey",
    "NM":"New Mexico",
    "NY":"New York",
    "NC":"North Carolina",
    "ND":"North Dakota",
    "OH":"Ohio",
    "OK":"Oklahoma",
    "OR":"Oregon",
    "PA":"Pennsylvania",
    "RI":"Rhode Island",
    "SC":"South Carolina",
    "SD":"South Dakota",
    "TN":"Tennessee",
    "TX":"Texas",
    "UT":"Utah",
    "VT":"Vermont",
    "VA":"Virginia",
    "WA":"Washington",
    "WV":"West Virginia",
    "WI":"Wisconsin",
    "WY":"Wyoming"
}

In [16]:
restaurants_clean["state_name"] = (
    restaurants_clean["province"]
    .map(state_map)
)

In [17]:
merged = restaurants_clean.merge(
    cities[
        ["city", "state_name", "population"]
    ],
    on=["city", "state_name"],
    how="left"
)

In [18]:
print(merged.shape)

print(
    merged["population"]
    .isna()
    .sum()
)

(10002, 7)
775


In [27]:
merged["population"].isna().sum()

np.int64(0)

In [22]:
print("Before:", merged.shape)

merged = merged.dropna(
    subset=["population"]
)

print("After:", merged.shape)

Before: (10002, 7)
After: (9227, 7)


In [24]:
missing_cities = merged[
    merged["population"].isna()
][["city", "state_name"]]

missing_cities.drop_duplicates().head(50)

,city,state_name


In [25]:
merged.to_csv(
    "../data/processed/merged_restaurant_data.csv",
    index=False
)

print("Saved successfully")

Saved successfully


In [26]:
merged.info()

<class 'pandas.DataFrame'>
Index: 9227 entries, 1 to 10001
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        9227 non-null   str    
 1   city        9227 non-null   str    
 2   province    9227 non-null   str    
 3   latitude    9227 non-null   float64
 4   longitude   9227 non-null   float64
 5   state_name  9227 non-null   str    
 6   population  9227 non-null   float64
dtypes: float64(3), str(4)
memory usage: 576.7 KB


In [28]:
merged["population"] = merged["population"].astype(int)

In [29]:
merged.to_csv(
    "../data/processed/merged_restaurant_data.csv",
    index=False
)